# Review Live/Paper Trading Run

Visualize a paper or live trading run using the same charts as the backtest
notebooks — TradingView Lightweight Charts with trade markers, equity curve,
stats summary.

Data comes from PostgreSQL (written by PersistenceActor during trading)
and candle data from the ParquetDataCatalog.

**Prerequisites:**
- PostgreSQL running (`docker compose up -d postgres`)
- At least one completed or active trading run with fills in the database
- Candle data in `data/catalog/` covering the run's time period

In [ ]:
# ── Cell 1: Imports + config ───────────────────────────────────────

import asyncpg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.config.settings import get_settings
from src.core import bar_type_str
from charts import generate_backtest_html, plot_pnl_heatmap

settings = get_settings()
CATALOG_PATH = "../data/catalog"
BAR_INTERVAL = "5m"  # Match what the strategy was trading on
# Infer MA periods from strategy config if available
# Chart uses these for overlay lines only, not for signal generation
FAST_PERIOD = 5
SLOW_PERIOD = 45
MA_TYPE = "EMA"  # or "SMA" depending on your strategy

print(f"DB: {settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}")
print("Imports OK")

In [ ]:
# ── Cell 2: List recent runs ───────────────────────────────────────

async def _list_runs():
    conn = await asyncpg.connect(settings.postgres_dsn)
    try:
        rows = await conn.fetch("""
            SELECT
                r.id,
                r.strategy_id,
                r.instrument_id,
                r.run_mode,
                r.started_at,
                r.stopped_at,
                (SELECT COUNT(*) FROM order_fills f WHERE f.run_id = r.id) AS fills,
                (SELECT COUNT(*) FROM positions p WHERE p.run_id = r.id) AS positions
            FROM strategy_runs r
            ORDER BY r.started_at DESC
            LIMIT 20
        """)
        return pd.DataFrame([dict(r) for r in rows])
    finally:
        await conn.close()

runs_df = await _list_runs()

if runs_df.empty:
    print("No runs found in database.")
else:
    print(f"Found {len(runs_df)} recent runs:\n")
    for i, row in runs_df.iterrows():
        status = "running" if row["stopped_at"] is None else "stopped"
        print(
            f"  [{i}] {row['strategy_id']}  {row['instrument_id']}  "
            f"{row['run_mode']}  {status}  "
            f"fills={row['fills']}  positions={row['positions']}  "
            f"started={row['started_at']:%Y-%m-%d %H:%M}"
        )

In [ ]:
# ── Cell 3: Select run + load data ─────────────────────────────────
#
# Change PICK to select a different run from the list above.

PICK = 0

run = runs_df.iloc[PICK]
RUN_ID = run["id"]
STRATEGY_ID = run["strategy_id"]
INSTRUMENT_ID = run["instrument_id"]
RUN_MODE = run["run_mode"]

print(f"Selected run: {STRATEGY_ID} on {INSTRUMENT_ID} ({RUN_MODE})")
print(f"Run ID: {RUN_ID}")
print(f"Started: {run['started_at']}")
print(f"Stopped: {run['stopped_at'] or 'still running'}")
print(f"Fills: {run['fills']}, Positions: {run['positions']}")


async def _load_run_data(run_id):
    conn = await asyncpg.connect(settings.postgres_dsn)
    try:
        fills_rows = await conn.fetch("""
            SELECT ts, strategy_id, instrument_id, client_order_id,
                   order_side, last_qty, last_px, commission,
                   commission_currency, liquidity_side
            FROM order_fills
            WHERE run_id = $1
            ORDER BY ts
        """, run_id)

        positions_rows = await conn.fetch("""
            SELECT ts_opened, ts_closed, strategy_id, instrument_id,
                   position_id, entry_side, peak_qty,
                   avg_px_open, avg_px_close,
                   realized_pnl, realized_return, currency, duration_ns
            FROM positions
            WHERE run_id = $1
            ORDER BY ts_closed
        """, run_id)

        snapshots_rows = await conn.fetch("""
            SELECT ts, venue, currency, balance_total, balance_free, balance_locked
            FROM account_snapshots
            WHERE run_id = $1
            ORDER BY ts
        """, run_id)

        return (
            pd.DataFrame([dict(r) for r in fills_rows]),
            pd.DataFrame([dict(r) for r in positions_rows]),
            pd.DataFrame([dict(r) for r in snapshots_rows]),
        )
    finally:
        await conn.close()

raw_fills, raw_positions, raw_snapshots = await _load_run_data(RUN_ID)
print(f"\nLoaded: {len(raw_fills)} fills, {len(raw_positions)} positions, {len(raw_snapshots)} snapshots")

In [ ]:
# ── Cell 4: Reshape data for charts.py ─────────────────────────────
#
# charts.py functions expect specific column names from NT's in-memory
# reports. PG schema uses slightly different names. Map them here.

# ── Fills ──
# _parse_fills looks for: last_px, order_side, ts_last (int64 ns), last_qty
# _fills_to_markers looks for: ts_last, side, avg_px, filled_qty
fills_df = raw_fills.copy()
if not fills_df.empty:
    # Convert TIMESTAMPTZ to nanosecond int64 (what NT uses internally)
    fills_df["ts_last"] = pd.to_datetime(fills_df["ts"], utc=True).astype("int64")
    # Add column aliases for charts.py compatibility
    fills_df["side"] = fills_df["order_side"]
    fills_df["avg_px"] = fills_df["last_px"]
    fills_df["filled_qty"] = fills_df["last_qty"]
    print(f"Fills: {len(fills_df)} rows, {fills_df['ts'].min():%Y-%m-%d} → {fills_df['ts'].max():%Y-%m-%d}")
else:
    print("No fills found.")

# ── Positions ──
# _positions_to_rows looks for: ts_opened, ts_closed, entry, peak_qty,
#   avg_px_open, avg_px_close, realized_pnl, realized_return
positions_df = raw_positions.copy()
if not positions_df.empty:
    # Rename entry_side → entry (charts.py expects "entry")
    positions_df["entry"] = positions_df["entry_side"]
    # realized_pnl is NUMERIC in PG — _parse_money_str handles plain numbers
    print(f"Positions: {len(positions_df)} closed trades")
else:
    print("No closed positions found.")

# ── Account snapshots ──
snapshots_df = raw_snapshots.copy()
if not snapshots_df.empty:
    snapshots_df["ts"] = pd.to_datetime(snapshots_df["ts"], utc=True)
    print(f"Snapshots: {len(snapshots_df)} balance records")
else:
    print("No account snapshots found.")

In [ ]:
# ── Cell 5: Load bars from catalog ─────────────────────────────────
#
# The TVLC chart needs candle data. Load from catalog for the run's
# time period. If bars aren't available, run your fetch script first:
#   python scripts/fetch_hl_candles.py --coins BTC --intervals 1h

BAR_TYPE_STR = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)

catalog = ParquetDataCatalog(CATALOG_PATH)

try:
    bars = catalog.bars(bar_types=[BAR_TYPE_STR])
    print(f"Loaded {len(bars):,} bars for {BAR_TYPE_STR}")

    # Filter to run's time range (with some padding for chart context)
    if not fills_df.empty:
        run_start = fills_df["ts"].min()
        run_end = fills_df["ts"].max()

        # Add padding: 5% of run duration on each side for chart context
        run_duration = run_end - run_start
        pad = run_duration * 0.05
        filter_start = int((run_start - pad).timestamp() * 1e9)
        filter_end = int((run_end + pad).timestamp() * 1e9)

        bars = [b for b in bars if filter_start <= b.ts_event <= filter_end]
        print(f"Filtered to {len(bars):,} bars covering run period")

except Exception as e:
    bars = []
    print(f"Could not load bars: {e}")
    print(f"Run your fetch script to get candle data for {INSTRUMENT_ID}")

In [ ]:
# ── Cell 6: Equity curve from account snapshots ────────────────────

if not snapshots_df.empty:
    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor("#131722")
    ax.set_facecolor("#131722")

    ax.plot(
        snapshots_df["ts"],
        snapshots_df["balance_total"].astype(float),
        color="#2196f3",
        linewidth=1.5,
    )

    # Starting balance reference line
    start_balance = float(snapshots_df["balance_total"].iloc[0])
    ax.axhline(start_balance, color="#787b86", linewidth=0.5, linestyle="--", alpha=0.5)

    ax.set_title(
        f"Equity Curve — {STRATEGY_ID} on {INSTRUMENT_ID} ({RUN_MODE})",
        color="#d1d4dc", fontsize=13,
    )
    ax.set_xlabel("Time", color="#d1d4dc")
    ax.set_ylabel(f"Balance ({snapshots_df['currency'].iloc[0]})", color="#d1d4dc")
    ax.tick_params(colors="#d1d4dc")
    ax.grid(True, alpha=0.1)

    # Annotate final balance
    final_balance = float(snapshots_df["balance_total"].iloc[-1])
    pnl = final_balance - start_balance
    pnl_pct = pnl / start_balance * 100
    color = "#26a69a" if pnl >= 0 else "#ef5350"
    ax.annotate(
        f"  {final_balance:,.2f} ({pnl:+,.2f} / {pnl_pct:+.1f}%)",
        xy=(snapshots_df["ts"].iloc[-1], final_balance),
        color=color, fontsize=11, fontweight="bold",
        va="center",
    )

    plt.tight_layout()
    plt.show()
else:
    print("No account snapshots — equity curve not available.")

In [ ]:
# ── Cell 7: Trade stats summary ───────────────────────────────────

if not positions_df.empty:
    pnls = positions_df["realized_pnl"].astype(float)
    returns = positions_df["realized_return"].astype(float)

    winners = pnls[pnls > 0]
    losers = pnls[pnls < 0]
    gross_win = winners.sum()
    gross_loss = abs(losers.sum())

    currency = positions_df["currency"].iloc[0] or "USD"

    print(f"{'═' * 50}")
    print(f"  TRADE SUMMARY — {STRATEGY_ID} ({RUN_MODE})")
    print(f"  {INSTRUMENT_ID}")
    print(f"{'═' * 50}")
    print(f"  Total trades:    {len(pnls)}")
    print(f"  Winners:         {len(winners)} ({len(winners)/len(pnls)*100:.0f}%)")
    print(f"  Losers:          {len(losers)} ({len(losers)/len(pnls)*100:.0f}%)")
    print(f"  Total PnL:       {pnls.sum():,.2f} {currency}")
    print(f"  Avg winner:      {winners.mean():,.2f} {currency}" if len(winners) > 0 else "")
    print(f"  Avg loser:       {losers.mean():,.2f} {currency}" if len(losers) > 0 else "")
    print(f"  Profit factor:   {gross_win / gross_loss:.2f}" if gross_loss > 0 else "  Profit factor:   ∞")
    print(f"  Avg return:      {returns.mean()*100:.2f}%")
    print(f"  Best trade:      {pnls.max():,.2f} {currency}")
    print(f"  Worst trade:     {pnls.min():,.2f} {currency}")

    if not snapshots_df.empty:
        start_bal = float(snapshots_df["balance_total"].iloc[0])
        final_bal = float(snapshots_df["balance_total"].iloc[-1])
        print(f"  Starting bal:    {start_bal:,.2f} {currency}")
        print(f"  Final bal:       {final_bal:,.2f} {currency}")
        print(f"  Return on acct:  {(final_bal - start_bal) / start_bal * 100:+.2f}%")

    print(f"{'═' * 50}")
else:
    print("No closed positions — no stats available.")

In [ ]:
# ── Cell 8: PnL per trade bar chart ───────────────────────────────

if not positions_df.empty:
    pnls = positions_df["realized_pnl"].astype(float).values

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor("#131722")

    # Left: PnL per trade (bar chart)
    ax = axes[0]
    ax.set_facecolor("#131722")
    colors = ["#26a69a" if p > 0 else "#ef5350" for p in pnls]
    ax.bar(range(len(pnls)), pnls, color=colors, width=1.0, edgecolor="none")
    ax.axhline(0, color="white", linewidth=0.5, alpha=0.3)
    ax.set_xlabel("Trade #", color="#d1d4dc")
    ax.set_ylabel("PnL", color="#d1d4dc")
    ax.set_title("PnL Per Trade", color="#d1d4dc")
    ax.tick_params(colors="#d1d4dc")

    # Right: Cumulative PnL
    ax = axes[1]
    ax.set_facecolor("#131722")
    cum_pnl = np.cumsum(pnls)
    ax.plot(cum_pnl, color="#2196f3", linewidth=1.5)
    ax.fill_between(
        range(len(cum_pnl)), cum_pnl, 0,
        where=cum_pnl >= 0, color="#26a69a", alpha=0.15,
    )
    ax.fill_between(
        range(len(cum_pnl)), cum_pnl, 0,
        where=cum_pnl < 0, color="#ef5350", alpha=0.15,
    )
    ax.axhline(0, color="white", linewidth=0.5, alpha=0.3)
    ax.set_xlabel("Trade #", color="#d1d4dc")
    ax.set_ylabel("Cumulative PnL", color="#d1d4dc")
    ax.set_title("Cumulative PnL", color="#d1d4dc")
    ax.tick_params(colors="#d1d4dc")

    plt.tight_layout()
    plt.show()
else:
    print("No positions to plot.")

In [ ]:
# ── Cell 9: TradingView Lightweight Charts HTML report ─────────
#
# Same interactive chart as the backtest notebook — candlesticks with
# buy/sell markers, stats bar, trade table with click-to-zoom.

if bars and not fills_df.empty:

    run_start_str = f"{run['started_at']:%Y%m%d}"

    # Determine starting capital from first account snapshot
    starting_capital = (
        float(snapshots_df["balance_total"].iloc[0])
        if not snapshots_df.empty
        else 10_000
    )

    path = generate_backtest_html(
        bars,
        fills_df,
        positions_df,
        fast_period=FAST_PERIOD,
        slow_period=SLOW_PERIOD,
        ma_type=MA_TYPE,
        instrument_label=INSTRUMENT_ID,
        bar_label=BAR_INTERVAL,
        starting_capital=starting_capital,
        result_filename=f"live_{STRATEGY_ID}_{INSTRUMENT_ID}_{run_start_str}",
    )
    print(f"Open in browser: {path}")
elif not bars:
    print("No bar data available. Run your candle fetch script first:")
    print(f"  python scripts/fetch_hl_candles.py --coins {INSTRUMENT_ID.split('-')[0]}")
else:
    print("No fills to overlay on chart.")


## Phase 2.5 — Cross-gate timing analysis

Compares the paper-trade per-bar signal stream (from `signal_events` in PG)
against a fresh backtest re-run of the same strategy over the same bar window.
Answers the roadmap's open question: *does the cross-gate fire at the same bar
in live as it does in backtest?*

**Modes:**
- `USE_SYNTHETIC_DATA = True` (default) — runs end-to-end on a synthetic 120-bar
  series with an injected 1-bar lag between paper and backtest. Useful for
  scaffold testing before Stage B real data exists.
- `USE_SYNTHETIC_DATA = False` — loads `signal_events` for the selected run
  (cells 1–4 above must have run) and re-runs `MACross` over the same bars.


In [ ]:
# === Phase 2.5 === Cell 10: Toggle synthetic vs real data

USE_SYNTHETIC_DATA = True   # flip to False once Stage B run_id exists

# Strategy hyperparameters used to BOTH the paper run AND the backtest re-run.
# When USE_SYNTHETIC_DATA=False, these MUST match what paper actually ran with.
P25_FAST = 10
P25_SLOW = 40
P25_MA_TYPE = 'EMA'

print(f'mode: {"SYNTHETIC" if USE_SYNTHETIC_DATA else "REAL"}')
print(f'params: fast={P25_FAST} slow={P25_SLOW} ma_type={P25_MA_TYPE}')


In [ ]:
# === Phase 2.5 === Cell 11: Build paper + backtest signal streams

from datetime import UTC as _UTC, datetime as _dt
from decimal import Decimal

from utils import (
    join_signal_streams,
    load_signal_events,
    run_backtest_signal_stream,
)
from src.core import get_venue_config

if USE_SYNTHETIC_DATA:
    # Synthetic mode — build a 120-bar BTC perp series, run two backtests
    # with the same params, then inject a 1-bar lag into the 'paper' stream
    # so divergence count is a known non-zero number.
    from nautilus_trader.model.data import Bar, BarType
    from nautilus_trader.model.objects import Price, Quantity
    from nautilus_trader.test_kit.providers import TestInstrumentProvider
    from src.core import bar_type_str

    _inst = TestInstrumentProvider.btcusdt_perp_binance()
    _bt_str = bar_type_str(str(_inst.id), '1d')
    _bar_type = BarType.from_str(_bt_str)
    _closes = (
        [100.0 - i for i in range(30)]
        + [70.0 + i for i in range(60)]
        + [130.0 - i for i in range(30)]
    )
    _base = int(_dt(2024, 1, 1, tzinfo=_UTC).timestamp() * 1e9)
    _one_day = 86_400 * 10**9
    _px_fmt = f'{{:.{int(_inst.price_precision)}f}}'
    _qty_fmt = f'{{:.{int(_inst.size_precision)}f}}'
    _bars = [
        Bar(_bar_type,
            Price.from_str(_px_fmt.format(c-0.5)),
            Price.from_str(_px_fmt.format(c+1.0)),
            Price.from_str(_px_fmt.format(c-1.0)),
            Price.from_str(_px_fmt.format(c)),
            Quantity.from_str(_qty_fmt.format(1.0)),
            _base + i * _one_day, _base + i * _one_day)
        for i, c in enumerate(_closes)
    ]
    _vc = get_venue_config('HYPERLIQUID_PERP')
    backtest_signals = run_backtest_signal_stream(
        instrument=_inst, bars=_bars, venue_config=_vc,
        fast_period=3, slow_period=5, ma_type='EMA', leverage=1,
    )
    # Inject a 1-bar lag for 'paper': shift ts forward by one bar so the
    # paper sees each signal one bar later than backtest. Demonstrates the
    # canonical Phase 2.5 finding shape end-to-end.
    paper_signals = backtest_signals.copy()
    paper_signals['ts'] = paper_signals['ts'] + pd.Timedelta(days=1)
else:
    # Real mode — load paper from PG, re-run backtest over the run's window.
    if 'RUN_ID' not in dir():
        raise RuntimeError(
            'Run cells 1-4 first to populate RUN_ID and the run window.'
        )
    paper_signals = await load_signal_events(settings.postgres_dsn, RUN_ID)
    if paper_signals.empty:
        raise RuntimeError(
            f'No signal_events rows for run {RUN_ID}. '
            'Was this run started with the post-Phase 2.5 schema?'
        )
    _vc = get_venue_config('HYPERLIQUID_PERP')
    backtest_signals = run_backtest_signal_stream(
        instrument=instrument,  # from cell 5
        bars=bars,              # from cell 5, already windowed
        venue_config=_vc,
        fast_period=P25_FAST,
        slow_period=P25_SLOW,
        ma_type=P25_MA_TYPE,
    )

print(f'paper stream:    {len(paper_signals):>6} rows')
print(f'backtest stream: {len(backtest_signals):>6} rows')


In [ ]:
# === Phase 2.5 === Cell 12: Join and summarize divergence

joined = join_signal_streams(paper_signals, backtest_signals)

n_total = len(joined)
n_divergent = int(joined['divergent'].sum())
div_pct = (n_divergent / n_total * 100) if n_total else 0.0

print('Phase 2.5 cross-gate alignment')
print('=' * 60)
print(f'Joined bars (present in both streams): {n_total}')
print(f'Paper-only bars (backtest did not see): {joined.attrs["paper_only_bars"]}')
print(f'Backtest-only bars (paper did not see): {joined.attrs["bt_only_bars"]}')
print(f'Divergent bars (paper signal != backtest signal): {n_divergent} ({div_pct:.1f}%)')
print()

# acted-row alignment — the bars where either side actually entered or flipped
either_acted = joined[joined['paper_acted'] | joined['bt_acted']]
agreed_acted = joined[joined['paper_acted'] & joined['bt_acted']]
print(f'Bars where either stream acted: {len(either_acted)}')
print(f'Bars where both streams acted on same bar: {len(agreed_acted)}')
if len(either_acted):
    print(f'Same-bar acted agreement: {len(agreed_acted) / len(either_acted) * 100:.1f}%')

joined.head(10)


In [ ]:
# === Phase 2.5 === Cell 13: Divergent-bar deep-dive

# Inspect the bars where paper and backtest disagree on the gate direction.
# These are the actionable rows — investigate fast/slow value gaps (NT
# EXTERNAL-aggregated trades vs catalog snapshot OHLCV).

div = joined[joined['divergent']].copy()
if div.empty:
    print('No divergent bars — paper and backtest gates align on every joined bar.')
else:
    div['fast_gap'] = (div['paper_fast'].astype(float) - div['bt_fast'].astype(float))
    div['slow_gap'] = (div['paper_slow'].astype(float) - div['bt_slow'].astype(float))
    div['fast_gap_bps'] = div['fast_gap'] / div['bt_fast'].astype(float) * 10_000
    div['slow_gap_bps'] = div['slow_gap'] / div['bt_slow'].astype(float) * 10_000
    print('Top divergent bars by fast-MA bps gap:')
    display(
        div.sort_values('fast_gap_bps', key=abs, ascending=False)
        [['ts', 'paper_signal', 'bt_signal',
          'paper_fast', 'bt_fast', 'fast_gap_bps',
          'paper_slow', 'bt_slow', 'slow_gap_bps']]
        .head(20)
    )


In [ ]:
# === Phase 2.5 === Cell 14: MA-value-gap histogram

# Distribution of (paper_fast - backtest_fast) and (paper_slow - backtest_slow)
# in basis points. A tight spike at 0 means NT's live EXTERNAL aggregation
# produces the same OHLCV as the catalog snapshot bars. Spread or skew
# indicates the upstream bar values diverged.

fast_bps = (
    (joined['paper_fast'].astype(float) - joined['bt_fast'].astype(float))
    / joined['bt_fast'].astype(float) * 10_000
).dropna()
slow_bps = (
    (joined['paper_slow'].astype(float) - joined['bt_slow'].astype(float))
    / joined['bt_slow'].astype(float) * 10_000
).dropna()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(fast_bps, bins=40, color='#4C9AFF', edgecolor='white')
ax1.axvline(0, color='red', linestyle='--', linewidth=1)
ax1.set_title(f'fast_MA gap (paper - backtest), bps  |  median={fast_bps.median():.2f}')
ax1.set_xlabel('bps')
ax1.set_ylabel('bars')
ax2.hist(slow_bps, bins=40, color='#36B37E', edgecolor='white')
ax2.axvline(0, color='red', linestyle='--', linewidth=1)
ax2.set_title(f'slow_MA gap (paper - backtest), bps  |  median={slow_bps.median():.2f}')
ax2.set_xlabel('bps')
plt.tight_layout()
plt.show()


In [ ]:
# === Phase 2.5 === Cell 15: Verdict snapshot for PHASE_2_5_VERDICT.md

# One-pass summary string that can be pasted into docs/PHASE_2_5_VERDICT.md.
# Numbers come from cell 12; this is just the formatting.

verdict = f"""## Cross-gate timing — measured

| Metric | Value |
|---|---|
| Joined bars | {n_total} |
| Paper-only bars | {joined.attrs['paper_only_bars']} |
| Backtest-only bars | {joined.attrs['bt_only_bars']} |
| Divergent bars (signal disagreement) | {n_divergent} ({div_pct:.1f}%) |
| Bars where either side acted | {len(either_acted)} |
| Same-bar acted agreement | {len(agreed_acted)}/{len(either_acted)} "
    f"({(len(agreed_acted) / len(either_acted) * 100 if len(either_acted) else 0):.1f}%) |
| fast_MA gap median (bps) | {fast_bps.median():.3f} |
| fast_MA gap p95 (bps) | {fast_bps.abs().quantile(0.95):.3f} |
| slow_MA gap median (bps) | {slow_bps.median():.3f} |
| slow_MA gap p95 (bps) | {slow_bps.abs().quantile(0.95):.3f} |
"""
print(verdict)
